# 실제 영상 채점 노트북

새 영상(또는 이미 추출된 키포인트 `.npy`) 하나를 넣으면, 지금까지 만든 파이프라인
전체(좌표 추출 → 전처리 → 특징 추출 → 채점)를 그대로 태워서 **rep별 점수와 어떤 부분이
문제인지**까지 보여줍니다.

전체 흐름:
```
INPUT_PATH(.mp4 또는 .npy)
  -> (영상이면) BlazePoseExtractor로 좌표 추출
  -> build_dataset.process_source_file()  # 마스킹/보간 -> 정규화 -> 신뢰도 높은 쪽 선택
                                            #  -> rep 슬라이싱 -> 위상정규화 -> 특징 추출
  -> scoring_model.load_reference() + score_rep()  # 학습된 기준 통계로 채점
  -> rep별 점수 표 + 막대그래프 + 자연어 피드백
```

> **사전 준비물**: `models/{EXERCISE}_reference.npz`(scoring_model.train으로 미리 학습해둔 파일)가
> 있어야 합니다. 영상을 직접 넣으실 거면 `models/pose_landmarker_full.task`(BlazePose 모델)도 필요합니다.

## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.pose_extraction.blazepose_extractor import BlazePoseExtractor, ExtractionConfig
from src.scoring_model.score_reps import score_video, load_template

plt.rcParams["figure.figsize"] = (9, 4)
np.set_printoptions(precision=3, suppress=True)

## 1. 입력 설정

`INPUT_PATH`에 영상(`.mp4` 등) 또는 이미 추출된 키포인트(`.npy`)를 넣으세요.
영상이면 자동으로 좌표 추출부터 진행하고, `.npy`면 바로 전처리로 넘어갑니다.

In [ ]:
INPUT_PATH = Path("data/raw/pushup_new_session.mp4")  # 영상 또는 .npy 경로로 바꾸세요
EXERCISE = "pushup"
MODEL_DIR = project_root / "models"
PROCESSED_DIR = project_root / "data" / "processed"  # build_dataset이 저장한 templates.npz 위치
POSE_MODEL_PATH = "models/pose_landmarker_full.task"  # 영상을 직접 넣을 때만 필요

MIN_DISTANCE = 30       # rep_slicer.py 기본값과 동일 (과도 분절 방지)
MIN_REP_FRAMES = 30

reference_path = MODEL_DIR / f"{EXERCISE}_reference.npz"
if not reference_path.exists():
    raise FileNotFoundError(
        f"{reference_path} 가 없습니다. 먼저 scoring_model.train으로 '{EXERCISE}' 기준 모델을 학습해두세요:\n"
        f"  python -m src.scoring_model.train --processed-dir data/processed --exercise {EXERCISE} --output-dir models"
    )

# 학습 때 build_dataset()이 만든 DTW 템플릿을 그대로 불러온다 (채점 시 새로 만들면 기준이 어긋남)
template = load_template(PROCESSED_DIR, EXERCISE)
print(f"'{EXERCISE}' 템플릿 로드 완료 (길이 {len(template)})")

## 2. 좌표 추출 (영상 입력일 때만)

`INPUT_PATH`가 `.npy`면 이 단계는 건너뜁니다.

In [ ]:
if INPUT_PATH.suffix.lower() == ".npy":
    keypoints_path = INPUT_PATH
    print(f"이미 추출된 키포인트 사용: {keypoints_path}")
else:
    config = ExtractionConfig(model_path=POSE_MODEL_PATH, include_z=False, target_fps=None)
    with BlazePoseExtractor(config) as extractor:
        print(f"🔄 좌표 추출 중: {INPUT_PATH.name}")
        keypoints, meta = extractor.extract_from_video(INPUT_PATH)

    # build_dataset이 기대하는 {exercise}_{video_id}_keypoints.npy 규칙에 맞춰 저장
    # (원본 영상 파일명을 그대로 리네이밍할 필요 없이 EXERCISE를 접두사로 강제)
    keypoints_path = INPUT_PATH.parent / f"{EXERCISE}_{INPUT_PATH.stem}_keypoints.npy"
    np.save(keypoints_path, keypoints)
    print(f"✅ 추출 완료: shape={keypoints.shape}, 검출률={meta['detected_ratio']:.1%}")
    print(f"저장: {keypoints_path}")

## 3. 전처리 + 특징 추출 + 채점

`score_video()` 하나로 마스킹/보간 → 정규화 → 신뢰도 높은 쪽 선택 → rep 슬라이싱
→ **DTW 위상 정규화(학습 때 쓴 템플릿 재사용)** → 특징 추출 → 채점까지 전부 처리됩니다
(build_dataset과 동일한 로직 + 저장된 ReferenceStats로 바로 채점까지 이어짐).

In [ ]:
results = score_video(
    keypoints_path=keypoints_path,
    exercise=EXERCISE,
    template=template,
    model_dir=MODEL_DIR,
    min_distance=MIN_DISTANCE,
    min_rep_frames=MIN_REP_FRAMES,
)

results_df = pd.DataFrame(results)
print(f"탐지된 rep 개수: {len(results_df)}")
print(f"평균 점수: {results_df['score'].mean():.1f} / 100")
print(f"최저 점수 rep: {results_df.loc[results_df['score'].idxmin(), 'rep_idx']}"
      f" (score={results_df['score'].min():.1f})")
results_df[["rep_idx", "score", "distance", "top_issues"]]

## 5. 시각화

In [ ]:
fig, ax = plt.subplots()
colors = ["tab:red" if s < 50 else "tab:orange" if s < 75 else "tab:green" for s in results_df["score"]]
ax.bar(results_df["rep_idx"], results_df["score"], color=colors)
ax.axhline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("rep_idx"); ax.set_ylabel("score"); ax.set_ylim(0, 100)
ax.set_title(f"{EXERCISE} rep별 점수 (빨강<50, 주황<75, 초록>=75)")
plt.show()

## 6. 자연어 피드백

`top_issues`(어느 특징이 얼마나 벗어났는지)를 사람이 읽을 수 있는 문장으로 바꿔서,
가장 점수가 낮았던 rep들에 대해 무엇이 문제였는지 보여줍니다.

In [ ]:
FEEDBACK_TEMPLATES = {
    "min_angle_elbow": "팔꿈치가 충분히 굽혀지지 않았습니다 (가동범위 부족)",
    "max_angle_elbow": "팔을 끝까지 펴지 않았습니다",
    "rom_elbow": "팔의 전체 가동범위가 정상 범위를 벗어났습니다",
    "min_angle_knee": "무릎이 충분히 굽혀지지 않았습니다 (스쿼트 깊이 부족)",
    "max_angle_knee": "무릎을 끝까지 펴지 않았습니다",
    "rom_knee": "무릎의 전체 가동범위가 정상 범위를 벗어났습니다",
}


def describe_issue(feature_name: str, z_score: float) -> str:
    base = FEEDBACK_TEMPLATES.get(feature_name, f"'{feature_name}' 값이 평소와 다릅니다")
    direction = "평균보다 큼" if z_score > 0 else "평균보다 작음"
    return f"{base} (편차: {z_score:+.2f}, {direction})"


low_score_reps = results_df.sort_values("score").head(5)
print("=== 점수가 낮은 rep TOP 5 피드백 ===\n")
for _, row in low_score_reps.iterrows():
    print(f"[rep {row['rep_idx']}] score={row['score']:.1f}")
    for feature_name, z_score in row["top_issues"]:
        print(f"   - {describe_issue(feature_name, z_score)}")
    print()
